In [5]:
import pandas as pd
import glob
import os

def merge_csv_files(file_paths, output_path="merged_result.csv", fill_value=pd.NA):
    """
    拼接多个CSV文件，合并相同code的行，并按顺序重新命名列
    解决空隙问题：用指定值填充缺失数据
    
    Args:
        file_paths: CSV文件路径列表
        output_path: 输出文件路径
        fill_value: 填充缺失值的数值，默认为0
    """
    all_dataframes = []
    
    # 读取所有CSV文件
    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            # 清理数据
            df = df.dropna(subset=['code'])
            df = df[df['code'].str.strip() != '']
            all_dataframes.append(df)
        except Exception as e:
            print(f"读取文件 {file_path} 时出错: {e}")
    
    if not all_dataframes:
        return None
    
    # 从第一个数据框开始合并
    merged_df = all_dataframes[0]
    
    # 逐个合并其他数据框
    for df in all_dataframes[1:]:
        merged_df = pd.merge(merged_df, df, on='code', how='outer')
    
 
    
    # 按code列排序
    merged_df = merged_df.sort_values('code').reset_index(drop=True)
    
    # 重新命名列：保持code列不变，其他列按1,2,3,4...顺序命名
    new_columns = ['code']
    col_counter = 1
    for col in merged_df.columns:
        if col != 'code':
            new_columns.append("preempt_"+str(col_counter-1))
            col_counter += 1
    
    merged_df.columns = new_columns
    
    # 保存结果
    merged_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    return merged_df

def merge_from_directory(directory_path, pattern="*.csv", output_path="merged_result.csv", fill_value=0):
    """
    从指定目录读取所有CSV文件并合并
    
    Args:
        directory_path: 目录路径
        pattern: 文件匹配模式
        output_path: 输出文件路径
        fill_value: 填充缺失值的数值
    """
    file_paths = glob.glob(os.path.join(directory_path, pattern))
    
    if not file_paths:
        return None
    
    return merge_csv_files(file_paths, output_path, fill_value)

# 使用示例
if __name__ == "__main__":
    # 指定文件路径
    file_list = [
        "1/preempt.csv",
        "5/preempt.csv", 
        "25/preempt.csv"
    ]
    
    # 合并文件，用0填充缺失值
    merged_data = merge_csv_files(file_list, "preempt.csv", fill_value=0)
    
    # 如果需要用其他值填充，可以修改fill_value参数
    # merged_data = merge_csv_files(file_list, "merged_result.csv", fill_value=-1)